# Notebook 4 — Multi-Task Toxicity Prediction on Tox21

**Project:** Equivariant Graph Neural Networks with Chirality Encoding for Structure-Sensitive Molecular Property Prediction

**Goal of this notebook:**
1. Load the **Tox21** benchmark (7,831 molecules × 12 biological assay endpoints) via
   PyTorch Geometric's MoleculeNet pipeline.
2. Build **stereo-aware molecular graphs** from each molecule's SMILES using the same RDKit
   feature pipeline developed in Notebook 3 (atom-type + chirality + signed-volume nodes,
   bond-type + E/Z edges, one ETKDGv3 conformer per molecule).
3. Combine the **trained EGNN encoder** (transferred from Notebook 3) with **structural-alert
   Morgan fingerprints** in a hybrid architecture — the GNN captures learned geometry-aware
   patterns; the fingerprint branch captures human-interpretable toxic substructure alerts.
4. Train with **masked binary cross-entropy** to handle the sparse, imbalanced label matrix
   (many assay measurements are simply missing — we must not treat them as negatives).
5. Evaluate per-task **ROC-AUC** and macro-average AUC, with a per-task breakdown so each
   assay endpoint's difficulty is visible.
6. Save results as `results_tox21.json` for the final comparison notebook (NB6).

**Before running:** upload `egnn_stereo_best.pt` from Notebook 3 into this Colab session
(Files panel → upload). If it is missing the notebook will train the encoder from scratch,
which works fine but takes a bit longer.


In [ ]:
!pip install -q torch_geometric
!pip install -q rdkit


In [ ]:
import os
import json
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader as TorchDataLoader
from torch_geometric.datasets import MoleculeNet
from torch_geometric.data import Data, Batch
from torch_geometric.nn import global_mean_pool
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


## 1. Load Tox21 via MoleculeNet

PyTorch Geometric's `MoleculeNet(root, name='Tox21')` downloads and pre-processes the dataset
automatically. Each `data.y` is a `(1, 12)` tensor of binary labels (0 / 1 / NaN). NaN means
the molecule was not tested in that assay — we must mask these out during training and
evaluation, **never** treat them as negatives.

The 12 assay endpoints (standard MoleculeNet column order):
- **Nuclear receptor panel (7):** NR-AR, NR-AR-LBD, NR-AhR, NR-Aromatase, NR-ER, NR-ER-LBD,
  NR-PPAR-gamma
- **Stress-response panel (5):** SR-ARE, SR-ATAD5, SR-HSE, SR-MMP, SR-p53


In [ ]:
TOX21_TASKS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
]
NUM_TASKS = len(TOX21_TASKS)

DATA_ROOT = './data/Tox21'
tox21 = MoleculeNet(root=DATA_ROOT, name='Tox21')
print(f"Total Tox21 molecules: {len(tox21)}")
print(f"Sample data object: {tox21[0]}")
print(f"Label shape: {tox21[0].y.shape}")

# Count label distribution per task
labels_all = torch.stack([tox21[i].y.view(NUM_TASKS) for i in range(len(tox21))])
for t, name in enumerate(TOX21_TASKS):
    col = labels_all[:, t]
    valid = col[~torch.isnan(col)]
    pos = (valid == 1).sum().item()
    total = len(valid)
    print(f"  {name:20s}: {total:5d} valid,  {pos:4d} positive ({100*pos/max(total,1):.1f}%)")


## 2. Train / val / test split (80 / 10 / 10, scaffold-free)

We use a simple random split (same seed pattern as earlier notebooks) rather than scaffold
splitting — scaffold splitting is more rigorous for generalisation estimates but complicates the
cross-notebook comparison. The split indices are saved so NB6 can reload them.


In [ ]:
NUM_MOL = len(tox21)
perm = torch.randperm(NUM_MOL, generator=torch.Generator().manual_seed(SEED))
n_train = int(0.8 * NUM_MOL)
n_val   = int(0.1 * NUM_MOL)

train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train + n_val]
test_idx  = perm[n_train + n_val:]

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

torch.save({'train_idx': train_idx, 'val_idx': val_idx, 'test_idx': test_idx},
           'tox21_split.pt')


## 3. Stereo-aware graph builder (same as Notebook 3)

We regenerate molecules' 3D graphs from SMILES using RDKit ETKDGv3 + MMFF94 (one low-energy
conformer per molecule — more than one is unnecessary for classification and would triple the
already-substantial conformer-generation time on 7k+ molecules). Node features: atom-type
one-hot (5) + chirality one-hot (3) + signed volume (1) = **9 dims**. Edge features: bond-type
one-hot (4) + E/Z one-hot (3) = **7 dims**.


In [ ]:
ATOM_TYPES   = ['H', 'C', 'N', 'O', 'F', 'Cl', 'Br', 'S', 'P', 'I']
ATOM_TO_IDX  = {a: i for i, a in enumerate(ATOM_TYPES)}
BOND_TYPES   = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
                Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
CHIRALITY_TYPES    = ['none', 'R', 'S']
CHIRALITY_TO_IDX   = {c: i for i, c in enumerate(CHIRALITY_TYPES)}
STEREO_BOND_TYPES  = ['none', 'E', 'Z']
STEREO_BOND_TO_IDX = {c: i for i, c in enumerate(STEREO_BOND_TYPES)}

NODE_DIM = len(ATOM_TYPES) + len(CHIRALITY_TYPES) + 1   # 10 + 3 + 1 = 14
EDGE_DIM = len(BOND_TYPES) + len(STEREO_BOND_TYPES)     # 4 + 3 = 7

print(f"Node feature dim: {NODE_DIM}, Edge feature dim: {EDGE_DIM}")


In [ ]:
# NOTE: Tox21 molecules can contain heavier atoms (Cl, Br, S, P, I) absent in QM9.
# The atom vocabulary is therefore extended relative to Notebook 3.

def compute_signed_volume(mol, atom_idx, conf):
    atom = mol.GetAtomWithIdx(atom_idx)
    neighbor_idxs = sorted(n.GetIdx() for n in atom.GetNeighbors())
    if len(neighbor_idxs) < 3:
        return 0.0
    p0  = conf.GetAtomPosition(atom_idx)
    p0  = np.array([p0.x, p0.y, p0.z])
    pts = []
    for n_idx in neighbor_idxs[:3]:
        p = conf.GetAtomPosition(n_idx)
        pts.append(np.array([p.x, p.y, p.z]) - p0)
    return float(np.dot(pts[0], np.cross(pts[1], pts[2])))


def embed_one_conformer(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = SEED
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=5, params=params)
    if len(cids) == 0:
        return None, None
    energies = []
    for cid in cids:
        try:
            props = AllChem.MMFFGetMoleculeProperties(mol)
            ff    = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize()
            energies.append((cid, ff.CalcEnergy()))
        except Exception:
            energies.append((cid, float('inf')))
    energies.sort(key=lambda t: t[1])
    best_cid = energies[0][0]
    if not np.isfinite(energies[0][1]):
        return None, None
    return mol, best_cid


def mol_to_data_stereo(mol, conf_id, y_tensor):
    Chem.AssignStereochemistryFrom3D(mol, confId=conf_id)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    conf  = mol.GetConformer(conf_id)
    atoms = list(mol.GetAtoms())

    x = torch.zeros(len(atoms), NODE_DIM)
    for atom in atoms:
        i     = atom.GetIdx()
        a_idx = ATOM_TO_IDX.get(atom.GetSymbol())
        if a_idx is None:
            return None          # atom type outside vocabulary -> skip molecule
        x[i, a_idx] = 1.0
        cip = atom.GetProp('_CIPCode') if atom.HasProp('_CIPCode') else None
        x[i, len(ATOM_TYPES) + CHIRALITY_TO_IDX.get(cip, 0)] = 1.0
        if cip is not None:
            x[i, -1] = compute_signed_volume(mol, i, conf)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt       = bond.GetBondType()
        bt_onehot = [1.0 if bt == b else 0.0 for b in BOND_TYPES]
        st = bond.GetStereo()
        if   st == Chem.BondStereo.STEREOE: s_label = 'E'
        elif st == Chem.BondStereo.STEREOZ: s_label = 'Z'
        else:                               s_label = 'none'
        s_onehot = [0.0, 0.0, 0.0]
        s_onehot[STEREO_BOND_TO_IDX[s_label]] = 1.0
        full_attr = bt_onehot + s_onehot
        edge_index += [[i, j], [j, i]]
        edge_attr  += [full_attr, full_attr]

    if len(edge_index) == 0:
        return None

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)
    pos        = torch.tensor(conf.GetPositions(), dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                pos=pos, y=y_tensor.clone())


## 4. Structural-alert Morgan fingerprints

Morgan (ECFP4-style) fingerprints with radius=2 and 1024 bits. These capture substructure
patterns ("structural alerts") known to be associated with toxicity (e.g. epoxides, nitro
groups, Michael acceptors). Concatenating them with the GNN embedding gives the model an
explicit, chemist-interpretable toxicity vocabulary in addition to whatever the GNN learns from
geometry. This hybrid strategy typically out-performs either branch alone on small, sparse
datasets like Tox21.


In [ ]:
FP_RADIUS = 2
FP_NBITS  = 1024

def compute_morgan_fp(smiles):
    """Returns a 1024-bit Morgan fingerprint as a float32 numpy array,
    or None if the SMILES can't be parsed. Uses the canonical (no-H)
    molecule — consistent with how structural-alert matching is done."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=FP_RADIUS, nBits=FP_NBITS)
    return np.array(fp, dtype=np.float32)


## 5. Build (and cache) the Tox21 dataset

This is the slow step — ~7 800 RDKit conformer embeddings. Cached to
`tox21_graphs_cache.pkl` so a Colab disconnect doesn't force a rebuild.


In [ ]:
CACHE_PATH = 'tox21_graphs_cache.pkl'

def build_tox21_dataset(tox21_pyg, cache_path):
    if os.path.exists(cache_path):
        print(f"Loading cached dataset from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    samples  = []   # list of (Data, fp_array)
    skipped  = 0
    for i in tqdm(range(len(tox21_pyg)), desc='Building Tox21 3D graphs'):
        raw = tox21_pyg[i]
        smiles = getattr(raw, 'smiles', None)
        if smiles is None:
            skipped += 1
            continue

        mol, conf_id = embed_one_conformer(smiles)
        if mol is None:
            skipped += 1
            continue

        y = raw.y.view(NUM_TASKS).float()   # (12,)  — may contain NaNs

        data = mol_to_data_stereo(mol, conf_id, y)
        if data is None:
            skipped += 1
            continue

        fp = compute_morgan_fp(smiles)
        if fp is None:
            skipped += 1
            continue

        samples.append((data, fp))

    print(f"Built {len(samples)} molecules, skipped {skipped}.")
    with open(cache_path, 'wb') as f:
        pickle.dump(samples, f)
    return samples


all_samples = build_tox21_dataset(tox21, CACHE_PATH)
print(f"Total usable molecules: {len(all_samples)}")


## 6. Dataset wrapper and collate function


In [ ]:
class Tox21Dataset(torch.utils.data.Dataset):
    def __init__(self, samples, indices):
        self.samples = [samples[i] for i in indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        data, fp = self.samples[i]
        return data, torch.tensor(fp, dtype=torch.float)


def collate_tox21(batch):
    graphs, fps = zip(*batch)
    pyg_batch = Batch.from_data_list(list(graphs))
    fp_batch  = torch.stack(fps)                    # (B, 1024)
    return pyg_batch, fp_batch


# Re-index splits against the (possibly smaller) usable set
N_USABLE = len(all_samples)
perm2 = torch.randperm(N_USABLE, generator=torch.Generator().manual_seed(SEED))
n_tr  = int(0.8 * N_USABLE)
n_v   = int(0.1 * N_USABLE)

train_set = Tox21Dataset(all_samples, perm2[:n_tr].tolist())
val_set   = Tox21Dataset(all_samples, perm2[n_tr:n_tr + n_v].tolist())
test_set  = Tox21Dataset(all_samples, perm2[n_tr + n_v:].tolist())

print(f"Final split — Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

train_loader = TorchDataLoader(train_set, batch_size=64,  shuffle=True,  collate_fn=collate_tox21)
val_loader   = TorchDataLoader(val_set,   batch_size=128, shuffle=False, collate_fn=collate_tox21)
test_loader  = TorchDataLoader(test_set,  batch_size=128, shuffle=False, collate_fn=collate_tox21)


## 7. Model — Hybrid EGNN + Structural-Alert Fingerprint

```
SMILES ──► RDKit 3D graph ──► EGNNEncoder ──► graph embedding (64-d)
                                                        │
SMILES ──► Morgan FP (1024-d) ──► FP branch ──► FP embedding (64-d)
                                                        │
                                      concat (128-d) ──► 12-head MLP ──► 12 task logits
```

- **EGNNEncoder** — 4 layers of the same E(n)-equivariant message-passing as Notebooks 2/3.
  Input dims now match the extended Tox21 atom vocabulary (NODE_DIM=14, EDGE_DIM=7).
- **FP branch** — lightweight two-layer MLP that compresses the 1024-bit fingerprint to 64-d.
  These bits encode substructure patterns that chemists already know are toxic alerts.
- **Shared 12-head output MLP** — one sigmoid output per assay task.

Weight transfer from Notebook 3: if `egnn_stereo_best.pt` is present we attempt to load the
encoder weights. Because the atom vocabulary is wider here (14 vs 9 node features), only the
non-input-layer weights transfer; the embedding layer is re-initialised and fine-tuned.


In [ ]:
class EGNNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_feat_dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_feat_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, h, pos, edge_index, edge_attr):
        row, col   = edge_index
        rel_pos    = pos[row] - pos[col]
        dist_sq    = (rel_pos ** 2).sum(dim=-1, keepdim=True)
        edge_input = torch.cat([h[row], h[col], dist_sq, edge_attr], dim=-1)
        m_ij       = self.edge_mlp(edge_input)

        coord_weight = self.coord_mlp(m_ij)
        agg_coord    = torch.zeros_like(pos)
        agg_coord.index_add_(0, row, rel_pos * coord_weight)
        deg = torch.zeros(pos.size(0), 1, device=pos.device)
        deg.index_add_(0, row, torch.ones_like(coord_weight))
        pos_new = pos + agg_coord / deg.clamp(min=1.0)

        agg_m = torch.zeros_like(h)
        agg_m.index_add_(0, row, m_ij)
        h_new = h + self.node_mlp(torch.cat([h, agg_m], dim=-1))
        return h_new, pos_new


class EGNNEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=4):
        super().__init__()
        self.embed  = nn.Linear(node_in_dim, hidden_dim)
        self.layers = nn.ModuleList(
            [EGNNLayer(hidden_dim, edge_in_dim) for _ in range(num_layers)]
        )

    def forward(self, x, pos, edge_index, edge_attr, batch):
        h = self.embed(x)
        for layer in self.layers:
            h, pos = layer(h, pos, edge_index, edge_attr)
        return global_mean_pool(h, batch)      # (B, hidden_dim)


class HybridTox21Model(nn.Module):
    """Hybrid EGNN + Morgan-fingerprint model for multi-task Tox21
    classification. GNN captures 3D/stereo geometry; FP branch captures
    interpretable structural alert patterns; both are concatenated and
    passed through a shared 12-output MLP.
    """
    def __init__(self, node_in_dim, edge_in_dim,
                 hidden_dim=64, num_layers=4,
                 fp_dim=FP_NBITS, num_tasks=NUM_TASKS):
        super().__init__()
        self.encoder = EGNNEncoder(node_in_dim, edge_in_dim, hidden_dim, num_layers)

        self.fp_branch = nn.Sequential(
            nn.Linear(fp_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, hidden_dim),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_tasks),
        )

    def forward(self, batch, fp):
        gnn_emb = self.encoder(batch.x, batch.pos, batch.edge_index,
                                batch.edge_attr, batch.batch)  # (B, H)
        fp_emb  = self.fp_branch(fp)                           # (B, H)
        combined = torch.cat([gnn_emb, fp_emb], dim=-1)        # (B, 2H)
        return self.classifier(combined)                        # (B, 12)  raw logits


In [ ]:
model_tox = HybridTox21Model(NODE_DIM, EDGE_DIM).to(device)

# ── Optional: transfer encoder weights from Notebook 3 ──────────────────
PRETRAINED_PATH = 'egnn_stereo_best.pt'
if os.path.exists(PRETRAINED_PATH):
    state = torch.load(PRETRAINED_PATH, map_location=device)
    # The checkpoint was saved from EGNNConformerEnsemble; encoder weights
    # are stored under the 'encoder.*' prefix.
    encoder_state = {k.replace('encoder.', ''): v
                     for k, v in state.items() if k.startswith('encoder.')}
    # Only load layers whose shape matches (non-input layers are identical).
    own_state = model_tox.encoder.state_dict()
    matched, skipped_keys = 0, 0
    for k, v in encoder_state.items():
        if k in own_state and own_state[k].shape == v.shape:
            own_state[k] = v
            matched += 1
        else:
            skipped_keys += 1
    model_tox.encoder.load_state_dict(own_state)
    print(f"Transferred {matched} encoder weight tensors, skipped {skipped_keys} "
          f"(shape mismatch — input layer re-initialised, all others transferred).")
else:
    print("egnn_stereo_best.pt not found — training encoder from scratch.")

total_params = sum(p.numel() for p in model_tox.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
print(model_tox)


## 8. Masked binary cross-entropy loss

Tox21's label matrix is very sparse: many entries are NaN (molecule not tested in that assay).
Treating NaN as 0 would massively inflate false negatives for common assay endpoints. Instead:
1. Compute `F.binary_cross_entropy_with_logits` element-wise (no reduction).
2. Build a boolean **mask** from `~torch.isnan(y)`.
3. Zero out NaN positions and average **only over valid positions**.

Additionally, Tox21 is heavily imbalanced (most molecules are non-toxic in most assays).  We
apply **positive-class weighting** (`pos_weight`) computed per-task from the training set so
the rare positive examples get amplified loss weight.


In [ ]:
def compute_pos_weights(loader, num_tasks, device):
    """Per-task ratio of negatives:positives in the training set.
    Passed as `pos_weight` to BCE loss to up-weight the minority class.
    """
    pos_counts = torch.zeros(num_tasks)
    neg_counts = torch.zeros(num_tasks)
    for batch, _ in loader:
        y = batch.y.view(-1, num_tasks)
        for t in range(num_tasks):
            col   = y[:, t]
            valid = col[~torch.isnan(col)]
            pos_counts[t] += (valid == 1).sum()
            neg_counts[t] += (valid == 0).sum()
    return (neg_counts / pos_counts.clamp(min=1.0)).to(device)


print("Computing per-task positive weights from training set...")
pos_weights = compute_pos_weights(train_loader, NUM_TASKS, device)
print("Per-task pos_weight (neg:pos ratio):")
for name, w in zip(TOX21_TASKS, pos_weights.tolist()):
    print(f"  {name:20s}: {w:.2f}")


In [ ]:
def masked_bce_loss(logits, y, pos_weights):
    """Binary cross-entropy averaged over non-NaN positions only.
    logits: (B, T), y: (B, T) with NaNs, pos_weights: (T,)
    """
    y = y.view(logits.shape)
    mask = ~torch.isnan(y)
    y_clean = y.clone()
    y_clean[~mask] = 0.0        # fill NaN with 0 to allow .backward()

    loss_per_element = F.binary_cross_entropy_with_logits(
        logits, y_clean, pos_weight=pos_weights.unsqueeze(0), reduction='none'
    )
    loss_masked = loss_per_element * mask.float()
    denom = mask.float().sum()
    if denom == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)
    return loss_masked.sum() / denom


## 9. Training utilities — ROC-AUC per task


In [ ]:
def train_one_epoch(model, loader, optimizer, pos_weights, device):
    model.train()
    total_loss, n = 0.0, 0
    for batch, fp in loader:
        batch = batch.to(device)
        fp    = fp.to(device)
        y     = batch.y.view(-1, NUM_TASKS).to(device)
        optimizer.zero_grad()
        logits = model(batch, fp)
        loss   = masked_bce_loss(logits, y, pos_weights)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        n          += batch.num_graphs
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    """Returns macro-average ROC-AUC and a per-task AUC dict."""
    model.eval()
    all_logits = []
    all_labels = []
    for batch, fp in loader:
        batch = batch.to(device)
        fp    = fp.to(device)
        y     = batch.y.view(-1, NUM_TASKS)
        logits = model(batch, fp).cpu()
        all_logits.append(logits)
        all_labels.append(y)

    all_logits = torch.cat(all_logits, dim=0).numpy()   # (N, 12)
    all_labels = torch.cat(all_labels, dim=0).numpy()   # (N, 12)

    per_task_auc = {}
    valid_aucs   = []
    for t, name in enumerate(TOX21_TASKS):
        y_t   = all_labels[:, t]
        p_t   = 1 / (1 + np.exp(-all_logits[:, t]))     # sigmoid
        mask  = ~np.isnan(y_t)
        if mask.sum() == 0 or len(np.unique(y_t[mask])) < 2:
            per_task_auc[name] = float('nan')
            continue
        auc = roc_auc_score(y_t[mask], p_t[mask])
        per_task_auc[name] = auc
        valid_aucs.append(auc)

    macro_auc = float(np.mean(valid_aucs)) if valid_aucs else float('nan')
    return macro_auc, per_task_auc


## 10. Train


In [ ]:
optimizer_tox = torch.optim.Adam(model_tox.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler_tox = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_tox, mode='max', factor=0.5, patience=5
)

N_EPOCHS_TOX   = 60
best_val_auc   = -1.0
history_tox    = {'train_loss': [], 'val_auc': []}

for epoch in range(1, N_EPOCHS_TOX + 1):
    train_loss = train_one_epoch(model_tox, train_loader, optimizer_tox, pos_weights, device)
    val_auc, _ = evaluate(model_tox, val_loader, device)
    scheduler_tox.step(val_auc)

    history_tox['train_loss'].append(train_loss)
    history_tox['val_auc'].append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model_tox.state_dict(), 'tox21_hybrid_best.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train loss: {train_loss:.4f} | "
              f"Val macro-AUC: {val_auc:.4f}")


## 11. Test evaluation — per-task ROC-AUC


In [ ]:
model_tox.load_state_dict(torch.load('tox21_hybrid_best.pt'))
test_auc, per_task_auc = evaluate(model_tox, test_loader, device)

print(f"\nTest macro-average ROC-AUC: {test_auc:.4f}")
print("\nPer-task ROC-AUC breakdown:")
print(f"{'Task':22s}  {'AUC':>6}")
print("-" * 32)
for name, auc in per_task_auc.items():
    flag = "  ← low" if (not np.isnan(auc)) and auc < 0.70 else ""
    print(f"{name:22s}  {auc:.4f}{flag}")


## 12. Fingerprint branch ablation

To probe the fingerprint branch, we zero out the fingerprint input at test time and re-evaluate
the *already-trained* model. The AUC drop shows how much the trained model relies on the
fingerprints. (This is an inference-time probe, not a retrained GNN-only model, so treat it as
indicative; Notebook 5 does proper retrained ablations.)


In [ ]:
@torch.no_grad()
def evaluate_no_fp(model, loader, device):
    """Evaluate with the fingerprint input replaced by zeros (ablation)."""
    model.eval()
    all_logits, all_labels = [], []
    for batch, fp in loader:
        batch    = batch.to(device)
        fp_zeros = torch.zeros_like(fp).to(device)   # ← zero out fingerprints
        y        = batch.y.view(-1, NUM_TASKS)
        logits   = model(batch, fp_zeros).cpu()
        all_logits.append(logits)
        all_labels.append(y)

    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    valid_aucs = []
    for t in range(NUM_TASKS):
        y_t  = all_labels[:, t]
        p_t  = 1 / (1 + np.exp(-all_logits[:, t]))
        mask = ~np.isnan(y_t)
        if mask.sum() == 0 or len(np.unique(y_t[mask])) < 2:
            continue
        valid_aucs.append(roc_auc_score(y_t[mask], p_t[mask]))
    return float(np.mean(valid_aucs)) if valid_aucs else float('nan')


auc_no_fp = evaluate_no_fp(model_tox, test_loader, device)
print(f"Test macro-AUC with   fingerprints: {test_auc:.4f}")
print(f"Test macro-AUC WITHOUT fingerprints (ablation): {auc_no_fp:.4f}")
print(f"Fingerprint branch contribution: {test_auc - auc_no_fp:+.4f}")


## 13. Save results


In [ ]:
results_tox21 = {
    'model_name':            'Hybrid_EGNN_StereoAware_Tox21',
    'dataset':               'Tox21',
    'num_tasks':             NUM_TASKS,
    'task_names':            TOX21_TASKS,
    'num_train':             len(train_set),
    'num_val':               len(val_set),
    'num_test':              len(test_set),
    'test_macro_roc_auc':    test_auc,
    'per_task_roc_auc':      per_task_auc,
    'ablation_no_fp_auc':    auc_no_fp,
    'fp_branch_contribution': test_auc - auc_no_fp,
}

with open('results_tox21.json', 'w') as f:
    json.dump(results_tox21, f, indent=2)

print(json.dumps(results_tox21, indent=2))


## 14. Training curve and per-task AUC bar chart


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training curve
axes[0].plot(history_tox['train_loss'], label='Train loss (masked BCE)')
axes[0].plot(history_tox['val_auc'],    label='Val macro-AUC', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_title('Tox21 Hybrid Model — Training Curve')
axes[0].legend()

# Per-task AUC bar chart
names = list(per_task_auc.keys())
aucs  = [v if not np.isnan(v) else 0.0 for v in per_task_auc.values()]
colors = ['#d62728' if a < 0.70 else '#2ca02c' for a in aucs]

axes[1].barh(names, aucs, color=colors)
axes[1].axvline(0.70, color='black', linestyle='--', linewidth=0.8, label='AUC = 0.70')
axes[1].axvline(test_auc, color='steelblue', linestyle='-', linewidth=1.2,
                label=f'Macro avg = {test_auc:.3f}')
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('ROC-AUC')
axes[1].set_title('Per-Task ROC-AUC (Tox21 test set)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## Done — files to keep for Notebook 6 (final comparison)

| File | Contents |
|---|---|
| `results_tox21.json` | Macro-AUC, per-task AUC, fingerprint ablation |
| `tox21_hybrid_best.pt` | Best model weights (optional — large) |
| `tox21_graphs_cache.pkl` | Pre-built 3D graphs (optional — regenerate if storage tight) |

**What to highlight in the viva:**
- **Masked BCE** is necessary — explain what happens if you don't mask (inflated accuracy, meaningless loss).
- **Positive weighting** handles class imbalance — cite the neg:pos ratios from §8.
- **Fingerprint ablation** in §12 gives concrete evidence that the structural-alert branch contributes.
- Per-task AUC varies a lot; some assays (e.g. SR-ARE, NR-AhR) are easier than others (NR-AR-LBD).

**Next up (Notebook 5):** BBBP binary classification — use the same stereo-aware encoder for
predicting blood-brain barrier permeability, a single-task binary problem, and compare against
this notebook's multi-task results in NB6.
